## Import libraries

In [1]:
from utils.dynamicRieszFunctions import estimateDynamicRiesz
from utils.dynamicRieszBradic import estimateBradicT1
import torch
import pandas as pd
import time

/Users/wisserutgers/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Generate data 


In [2]:
torch.manual_seed(1)

In [3]:
lower = 0.10
upper = 0.90

# Define logistic function
def logistic(x):
    return torch.exp(x) / (1 + torch.exp(x))

# Define a truncated logistic function
def truncated_logistic(x):
    return lower + (upper - lower) * logistic(x)

treatment_probability_func = logistic

In [4]:
# Parameters
N = 1000
tmax = 5
p1 = 2 # Amount of covariates (keep fixed for these tests)

# Coefficients
beta_pi1_0 = 0
beta_pi1_S1 = torch.tensor([1, 1], dtype=torch.float32).reshape(-1,1) 
beta_g0_0 = 1 * 4
beta_g0_S1 = torch.tensor([1, 1], dtype=torch.float32).reshape(-1,1) * 6
beta_g1_0 = -1 * 5
beta_g1_S1 = torch.tensor([-1, 1], dtype=torch.float32).reshape(-1,1) * 4

# Set seed
torch.manual_seed(123)

# Generate data
S1_all = torch.randn(N * tmax, p1)
pi1_all = treatment_probability_func(beta_pi1_0 + S1_all @ beta_pi1_S1).reshape(-1,1)
A1_all = torch.bernoulli(pi1_all).int().reshape(-1,1)

g_all = ((A1_all == 0).float() * (beta_g0_0 + S1_all @ beta_g0_S1 ) + 
         (A1_all == 1).float() * (beta_g1_0 + S1_all @ beta_g1_S1 ))
zeta_all = torch.randn(N * tmax,1)
Y_all = g_all + zeta_all

In [5]:
mu1_1_all = beta_g1_0 + S1_all @ (beta_g1_S1) 
mu1_0_all = beta_g0_0 + S1_all @ (beta_g0_S1)

theta0 = beta_g0_0
theta1 = beta_g1_0 
theta = theta1 - theta0
print(theta, theta1, theta0)

-9 -5 4


# Estimation:

In [6]:
folds = 2

time0 = time.time()

In [7]:
pred_theta = torch.zeros(tmax,5)
pred_sig = torch.zeros(tmax,5)

RR1 = torch.zeros(N,tmax,5)
RR2 = torch.zeros(N,tmax,5)

for t in range(0,tmax):

    S1 = S1_all[(t) * N : (t+1) * N, :]
    A1 = A1_all[(t) * N : (t+1) * N]
    Y = Y_all[(t) * N : (t+1) * N]

    X = ((S1))
    X_index = torch.tensor([S1.shape[1]-1])
    D = ((A1))

    # Counterfactual treatment of interest
    d = 1 * torch.ones(D.shape) 

    #---------------------------------------------------------------------------
    # Estimator 1: Oracle
    pi1 = pi1_all[(t) * N : (t+1) * N]
    mu1_1 = mu1_1_all[(t) * N : (t+1) * N]
    mu1_0 = mu1_0_all[(t) * N : (t+1) * N]
    if d[0] == 1:
        gamma = A1 / pi1  
        theta_hat = (gamma * Y - (gamma - 1) * mu1_1)
        pred_theta[t,0] = torch.mean( theta_hat )
        pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))
    else:
        gamma = (1 - A1) / (1 - pi1  )
        theta_hat = (gamma * Y - (gamma - 1) * mu1_0  )
        pred_theta[t,0] = torch.mean( theta_hat )
        pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))


    RR2[:,t,:1] = gamma

    #---------------------------------------------------------------------------
    # Estimator 2: Bradic
    # bradic_result = estimateBradicT1(Y,S1,A1, folds)
    # if d[0] == 1:
    #     pred_theta[t,1] = bradic_result[1]
    #     # pred_sig[t,1] = bradic_result[10]
    # elif d[0] == 0:
    #     pred_theta[t,1] = bradic_result[2]
    #     # pred_sig[t,1] = bradic_result[16]

    #---------------------------------------------------------------------------
    # Estimator 3: LASSO 

    point, sigma, RR1[:,t,2:3], RR2[:,t,2:3] = estimateDynamicRiesz(Y, X, D, d, X_index, folds, method_a = "LASSO", method_f = "LASSO_CV")
    pred_theta[t,2] = point
    pred_sig[t,2] = sigma

    #---------------------------------------------------------------------------
    # Estimator 4: RF 

    point, sigma, RR1[:,t,3:4], RR2[:,t,3:4] = estimateDynamicRiesz(Y, X, D, d, X_index, folds, method_a = "RF", method_f = "LASSO_CV")
    pred_theta[t,3] = point
    pred_sig[t,3] = sigma

    #---------------------------------------------------------------------------
    # Estimator 5: Net 


    point, sigma, RR1[:,t,4:5], RR2[:,t,4:5] = estimateDynamicRiesz(Y, X, D, d, X_index, folds, method_a = "Net", method_f = "Net")
    pred_theta[t,4] = point
    pred_sig[t,4] = sigma

    if t % 10 == 0:
        print("Time until iteration ",t, ": ", time.time() - time0)

print("Finished. Total time: ", time.time() - time0)


Time until iteration  0 :  21.00797390937805


KeyboardInterrupt: 

In [ ]:
%debug

ERROR:root:No traceback has been produced, nothing to debug.


## Compute statistics

### Right now: using theta1 (d = [1,1])

In [9]:
true_value = theta1

In [10]:
bias = torch.mean(pred_theta[:t+1,:] - true_value, 0)
rmse = torch.sqrt(torch.mean( (pred_theta[:t+1,:] - true_value) ** 2, 0))

ub = pred_theta[:t+1,:] + 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))
lb = pred_theta[:t+1,:] - 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))

coverage = torch.mean( ( (true_value >= lb) & (true_value <= ub) ).float() , 0 )
interval_length = torch.mean(ub - lb,0)

# Create a DataFrame
data = {
    "Method": ["Oracle", "Bradic", "LASSO", "RF", "Net"],
    "Bias": bias.ravel(),
    "RMSE": rmse.ravel(),
    "Coverage": coverage.ravel(),
    "Interval Length": interval_length.ravel()
}

In [11]:
df = pd.DataFrame(data)
print(df)

   Method      Bias      RMSE  Coverage  Interval Length
0  Oracle -0.126873  0.149212       1.0         0.732273
1  Bradic  5.000000  5.000000       0.0         0.000000
2   LASSO -0.126881  0.146714       1.0         0.966001
3      RF -0.259777  0.270558       1.0         0.940238
4     Net  0.927463  2.239412       0.8         0.589017


In [12]:
pred_theta

tensor([[-5.1811,  0.0000, -5.1545, -5.3077, -5.1990],
        [-5.1935,  0.0000, -5.2378, -5.3805, -5.1869],
        [-5.0169,  0.0000, -5.0620, -5.2315, -4.9921],
        [-5.0463,  0.0000, -5.0303, -5.1667, -4.9847],
        [-5.1966,  0.0000, -5.1497, -5.2125,  0.0000]])

## Compare riesz representers

In [ ]:
check_t = 0

# Compare all three:
pd.DataFrame(torch.hstack((RR2[:,check_t,0:1], RR2[:,check_t,2:3], RR2[:,check_t,3:4], RR2[:,check_t,4:5]))).head(50)
 
 # Currently: Estimation is terrible of NN

,0,1,2,3
0,0.000000,0.028912,0.000000,-0.019471
1,0.000000,0.027895,0.000000,-0.211746
2,1.364603,1.806815,1.528156,1.272575
3,2.819947,3.413102,2.679267,3.697784
4,0.000000,0.027895,0.000000,-0.021939
5,0.000000,0.028912,0.000000,-0.007804
6,1.125616,0.808429,1.331778,1.055714
7,1.465499,2.022603,1.434005,1.401846
8,0.000000,0.027895,0.000000,-0.164959
9,0.000000,0.028912,0.000000,0.174686
